# Part B — Feature Engineering and Outlier Analysis

**Project:** Academy Nova Course Cancellation Prediction  
**Group:** 51  
**Main metric:** ROC-AUC  

This notebook covers feature engineering and outlier analysis.

The goal of this stage is to transform the raw variables into more meaningful predictors while controlling overfitting and preserving generalization to the test set.

Important principles:
- Feature engineering must be based on business logic and EDA findings.
- Any transformation that learns values from the data must be fitted only on the training data.
- Outliers should not be removed blindly.
- Test rows must never be removed.
- New features should improve predictive signal without creating leakage.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import ID_COL, TARGET_COL, RANDOM_STATE
from src.data_loading import load_raw_data, validate_raw_data

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
train_df, test_df = load_raw_data()
validate_raw_data(train_df, test_df)

train_df.head()

Validating raw data...
Raw data validation passed.
Train shape: (63464, 29)
Test shape: (15866, 28)
Target positive rate: 0.4144


,Client_ID,Professionals_Count,Students_Count,Observers_Count,Course_Start_Date,Practical_Hours,Theory_Hours,Registration_Days_Before,Origin_Country,Catering_Package,Welcome_Gift_Type,Requested_Lab_Config,Assigned_Lab_Config,Prev_Course_Dropouts,Prev_Course_Attended,Pre_Course_Supports_Tickets,Physical_Course_Kits,Waiting_List_Days,Registration_Changes,Enrollment_Type,Lanyard_Color,Client_Category,Submission_Source,Returning_Client,Agent_ID,Company_ID,Payment_Terms,Daily_Tuition_Cost,Dropped_Course
0,13766,2,0.0,0,2015-07-01,0,2,257.0,PRT,Lunch Included,Branded Notebook,Standard PC (Windows),Standard PC (Windows),0,0,0,0.0,0,0,General Admission,Blue,Traditional IT & Telecomm,B2B Platforms & Resellers,0,219.0,NaN,Pay Upon Start,101.5,0
1,78660,1,0.0,0,2015-07-01,0,2,257.0,PRT,Lunch Included,Branded Notebook,Standard PC (Windows),Standard PC (Windows),0,0,0,0.0,0,1,General Admission,Blue,Traditional IT & Telecomm,B2B Platforms & Resellers,0,219.0,NaN,Pay Upon Start,80.0,0
2,51396,1,0.0,0,2015-07-01,0,2,257.0,PRT,Lunch Included,USB Drive,Standard PC (Windows),Standard PC (Windows),0,0,0,0.0,0,1,General Admission,Red,Traditional IT & Telecomm,B2B Platforms & Resellers,0,219.0,NaN,Pay Upon Start,80.0,0
3,34000,2,0.0,0,2015-07-01,0,2,257.0,PRT,Lunch Included,Branded Notebook,Standard PC (Windows),Standard PC (Windows),0,0,0,0.0,0,0,General Admission,Red,Traditional IT & Telecomm,B2B Platforms & Resellers,0,219.0,NaN,Pay Upon Start,101.5,0
4,69025,1,0.0,0,2015-07-01,0,2,257.0,PRT,Lunch Included,Branded Notebook,Standard PC (Windows),Standard PC (Windows),0,0,0,0.0,0,1,General Admission,Orange,Traditional IT & Telecomm,B2B Platforms & Resellers,0,219.0,NaN,Pay Upon Start,80.0,0


## 1. Feature Engineering Plan

Based on the EDA findings, we will create new features that represent business logic and operational patterns.

The main feature groups are:

1. Participant composition features  
2. Course workload and structure features  
3. Cost-related features  
4. Previous client/course history features  
5. Registration behavior features  
6. Missingness indicator features  
7. Lab configuration mismatch features  
8. Date-derived features  

Each feature should have a clear reason and should avoid target leakage.

In [3]:
def safe_divide(numerator, denominator):
    """
    Safely divide two values/Series.
    If denominator is 0 or missing, return 0.
    """
    return np.where(
        (denominator == 0) | pd.isna(denominator),
        0,
        numerator / denominator
    )

## 2. Create Engineered Features

The function below creates the same engineered features for both train and test sets.  
This is important because the model must receive identical columns during training and prediction.

In [5]:
def create_engineered_features(df):
    """
    Create business-driven engineered features.

    This function does not use the target column, so it can be safely applied
    to both train and test data.
    """
    df = df.copy()

    # ------------------------------------------------------------
    # Participant composition
    # ------------------------------------------------------------
    df["Total_Participants"] = (
        df["Professionals_Count"].fillna(0)
        + df["Students_Count"].fillna(0)
        + df["Observers_Count"].fillna(0)
    )

    df["Active_Participants"] = (
        df["Professionals_Count"].fillna(0)
        + df["Students_Count"].fillna(0)
    )

    df["Observer_Ratio"] = safe_divide(
        df["Observers_Count"].fillna(0),
        df["Total_Participants"]
    )

    df["Student_Ratio"] = safe_divide(
        df["Students_Count"].fillna(0),
        df["Total_Participants"]
    )

    df["Professional_Ratio"] = safe_divide(
        df["Professionals_Count"].fillna(0),
        df["Total_Participants"]
    )

    # ------------------------------------------------------------
    # Course structure / workload
    # ------------------------------------------------------------
    df["Total_Hours"] = (
        df["Practical_Hours"].fillna(0)
        + df["Theory_Hours"].fillna(0)
    )

    df["Practical_Ratio"] = safe_divide(
        df["Practical_Hours"].fillna(0),
        df["Total_Hours"]
    )

    df["Theory_Ratio"] = safe_divide(
        df["Theory_Hours"].fillna(0),
        df["Total_Hours"]
    )

    # ------------------------------------------------------------
    # Cost-related features
    # ------------------------------------------------------------
    df["Cost_Per_Hour"] = safe_divide(
        df["Daily_Tuition_Cost"],
        df["Total_Hours"]
    )

    df["Cost_Per_Participant"] = safe_divide(
        df["Daily_Tuition_Cost"],
        df["Total_Participants"]
    )

    # ------------------------------------------------------------
    # Previous history features
    # ------------------------------------------------------------
    df["Prev_Total_Courses"] = (
        df["Prev_Course_Dropouts"].fillna(0)
        + df["Prev_Course_Attended"].fillna(0)
    )

    df["Prev_Dropout_Rate"] = safe_divide(
        df["Prev_Course_Dropouts"].fillna(0),
        df["Prev_Total_Courses"]
    )

    df["Has_Previous_Course_History"] = (df["Prev_Total_Courses"] > 0).astype(int)

    # ------------------------------------------------------------
    # Support and registration behavior
    # ------------------------------------------------------------
    df["Support_Per_Participant"] = safe_divide(
        df["Pre_Course_Supports_Tickets"].fillna(0),
        df["Total_Participants"]
    )

    df["Changes_Per_Waiting_Day"] = safe_divide(
        df["Registration_Changes"].fillna(0),
        df["Waiting_List_Days"].fillna(0) + 1
    )

    df["High_Registration_Changes"] = (
        df["Registration_Changes"] >= df["Registration_Changes"].median()
    ).astype(int)

    df["Has_Support_Tickets"] = (
        df["Pre_Course_Supports_Tickets"] > 0
    ).astype(int)

    # ------------------------------------------------------------
    # Missingness indicators
    # ------------------------------------------------------------
    important_missing_cols = [
        "Company_ID",
        "Agent_ID",
        "Registration_Days_Before",
        "Requested_Lab_Config",
        "Physical_Course_Kits",
        "Enrollment_Type",
        "Submission_Source",
        "Payment_Terms",
        "Origin_Country",
        "Catering_Package",
        "Daily_Tuition_Cost",
        "Students_Count",
    ]

    for col in important_missing_cols:
        if col in df.columns:
            df[f"{col}_Was_Missing"] = df[col].isna().astype(int)

    df["Has_Company"] = df["Company_ID"].notna().astype(int)
    df["Has_Agent"] = df["Agent_ID"].notna().astype(int)

    # ------------------------------------------------------------
    # Lab configuration mismatch
    # ------------------------------------------------------------
    df["Lab_Config_Mismatch"] = (
        df["Requested_Lab_Config"].fillna("Missing")
        != df["Assigned_Lab_Config"].fillna("Missing")
    ).astype(int)

    # ------------------------------------------------------------
    # Date features
    # ------------------------------------------------------------
    df["Course_Start_Date"] = pd.to_datetime(df["Course_Start_Date"], errors="coerce")

    df["Course_Start_Year"] = df["Course_Start_Date"].dt.year
    df["Course_Start_Month"] = df["Course_Start_Date"].dt.month
    df["Course_Start_Quarter"] = df["Course_Start_Date"].dt.quarter
    df["Course_Start_DayOfWeek"] = df["Course_Start_Date"].dt.dayofweek
    df["Course_Start_WeekOfYear"] = df["Course_Start_Date"].dt.isocalendar().week.astype("Int64")
    df["Course_Start_IsWeekend"] = df["Course_Start_DayOfWeek"].isin([5, 6]).astype(int)

    return df

In [6]:
train_fe = create_engineered_features(train_df)
test_fe = create_engineered_features(test_df)

print("Original train shape:", train_df.shape)
print("Engineered train shape:", train_fe.shape)

print("Original test shape:", test_df.shape)
print("Engineered test shape:", test_fe.shape)

Original train shape: (63464, 29)
Engineered train shape: (63464, 67)
Original test shape: (15866, 28)
Engineered test shape: (15866, 66)


In [7]:
new_features = [col for col in train_fe.columns if col not in train_df.columns]

print(f"Number of new features: {len(new_features)}")
new_features

Number of new features: 38


['Total_Participants',
 'Active_Participants',
 'Observer_Ratio',
 'Student_Ratio',
 'Professional_Ratio',
 'Total_Hours',
 'Practical_Ratio',
 'Theory_Ratio',
 'Cost_Per_Hour',
 'Cost_Per_Participant',
 'Prev_Total_Courses',
 'Prev_Dropout_Rate',
 'Has_Previous_Course_History',
 'Support_Per_Participant',
 'Changes_Per_Waiting_Day',
 'High_Registration_Changes',
 'Has_Support_Tickets',
 'Company_ID_Was_Missing',
 'Agent_ID_Was_Missing',
 'Registration_Days_Before_Was_Missing',
 'Requested_Lab_Config_Was_Missing',
 'Physical_Course_Kits_Was_Missing',
 'Enrollment_Type_Was_Missing',
 'Submission_Source_Was_Missing',
 'Payment_Terms_Was_Missing',
 'Origin_Country_Was_Missing',
 'Catering_Package_Was_Missing',
 'Daily_Tuition_Cost_Was_Missing',
 'Students_Count_Was_Missing',
 'Has_Company',
 'Has_Agent',
 'Lab_Config_Mismatch',
 'Course_Start_Year',
 'Course_Start_Month',
 'Course_Start_Quarter',
 'Course_Start_DayOfWeek',
 'Course_Start_WeekOfYear',
 'Course_Start_IsWeekend']

In [8]:
feature_explanation = pd.DataFrame([
    {
        "feature_group": "Participant composition",
        "features": "Total_Participants, Active_Participants, Observer_Ratio, Student_Ratio, Professional_Ratio",
        "reason": "Cancellation risk may depend on course size and the composition of professionals, students, and observers."
    },
    {
        "feature_group": "Course structure",
        "features": "Total_Hours, Practical_Ratio, Theory_Ratio",
        "reason": "The workload and practical/theoretical balance may influence cancellation behavior."
    },
    {
        "feature_group": "Cost",
        "features": "Cost_Per_Hour, Cost_Per_Participant",
        "reason": "The same tuition cost may have different meaning depending on course length and number of participants."
    },
    {
        "feature_group": "Previous history",
        "features": "Prev_Total_Courses, Prev_Dropout_Rate, Has_Previous_Course_History",
        "reason": "Previous attendance/dropout history can indicate client reliability and future cancellation risk."
    },
    {
        "feature_group": "Registration behavior",
        "features": "Support_Per_Participant, Changes_Per_Waiting_Day, High_Registration_Changes, Has_Support_Tickets",
        "reason": "Frequent changes or support tickets may reflect uncertainty or operational friction before the course."
    },
    {
        "feature_group": "Missingness",
        "features": "Missing indicators, Has_Company, Has_Agent",
        "reason": "EDA showed that missingness itself can be predictive, especially for Company_ID and Agent_ID."
    },
    {
        "feature_group": "Lab configuration",
        "features": "Lab_Config_Mismatch",
        "reason": "A mismatch between requested and assigned lab configuration may increase dissatisfaction or cancellation risk."
    },
    {
        "feature_group": "Date",
        "features": "Month, Quarter, DayOfWeek, WeekOfYear, IsWeekend",
        "reason": "Date EDA showed temporal and weekday patterns in cancellation rates."
    },
])

feature_explanation

,feature_group,features,reason
0,Participant composition,"Total_Participants, Active_Participants, Obser...",Cancellation risk may depend on course size an...
1,Course structure,"Total_Hours, Practical_Ratio, Theory_Ratio",The workload and practical/theoretical balance...
2,Cost,"Cost_Per_Hour, Cost_Per_Participant",The same tuition cost may have different meani...
3,Previous history,"Prev_Total_Courses, Prev_Dropout_Rate, Has_Pre...",Previous attendance/dropout history can indica...
4,Registration behavior,"Support_Per_Participant, Changes_Per_Waiting_D...",Frequent changes or support tickets may reflec...
5,Missingness,"Missing indicators, Has_Company, Has_Agent",EDA showed that missingness itself can be pred...
6,Lab configuration,Lab_Config_Mismatch,A mismatch between requested and assigned lab ...
7,Date,"Month, Quarter, DayOfWeek, WeekOfYear, IsWeekend",Date EDA showed temporal and weekday patterns ...


In [9]:
engineered_numeric_features = [
    col for col in new_features
    if pd.api.types.is_numeric_dtype(train_fe[col])
]

invalid_values_check = pd.DataFrame({
    "feature": engineered_numeric_features,
    "train_missing": [train_fe[col].isna().sum() for col in engineered_numeric_features],
    "test_missing": [test_fe[col].isna().sum() for col in engineered_numeric_features],
    "train_inf": [np.isinf(train_fe[col].astype(float)).sum() for col in engineered_numeric_features],
    "test_inf": [np.isinf(test_fe[col].astype(float)).sum() for col in engineered_numeric_features],
})

invalid_values_check

,feature,train_missing,test_missing,train_inf,test_inf
0,Total_Participants,0,0,0,0
1,Active_Participants,0,0,0,0
2,Observer_Ratio,0,0,0,0
3,Student_Ratio,0,0,0,0
4,Professional_Ratio,0,0,0,0
5,Total_Hours,0,0,0,0
6,Practical_Ratio,0,0,0,0
7,Theory_Ratio,0,0,0,0
8,Cost_Per_Hour,79,1,0,0
9,Cost_Per_Participant,79,1,0,0


### Feature engineering initial conclusion

The engineered features add business-driven information from participant composition, course structure, cost, previous history, registration behavior, missingness, lab configuration, and date patterns.

These features are designed to improve predictive signal while avoiding target leakage.  
The next step is to analyze outliers and then evaluate whether these features improve cross-validation AUC during modeling.